# Analise de Dados - Camada Gold
Este notebook responde as perguntas de negocio e gera a tabela agregada da camada Gold (VIEW).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import banco
import warnings
warnings.filterwarnings('ignore')

conexao = banco.conectar()
sns.set_theme(style='whitegrid')

### Parte 1: Camada Silver - Análises Diretas

In [ ]:
# 1. Os 5 órgãos com maior custo total?
q1 = "SELECT nome_orgao_superior, SUM(valor_total) as custo FROM silver_viagem GROUP BY nome_orgao_superior ORDER BY custo DESC LIMIT 5"
df1 = pd.read_sql(q1, conexao)
display(df1)
plt.figure(figsize=(10,4))
sns.barplot(data=df1, y='nome_orgao_superior', x='custo', palette='viridis')
plt.title('Top 5 Órgãos com Maior Custo Total')
plt.xlabel('Custo Total (R$)')
plt.ylabel('Órgão Superior')
plt.show()

# 2. Os 3 destinos com maior custo médio por viagem?
q2 = "SELECT destinos, AVG(valor_total) as custo_medio FROM silver_viagem GROUP BY destinos ORDER BY custo_medio DESC LIMIT 3"
df2 = pd.read_sql(q2, conexao)
display(df2)
plt.figure(figsize=(10,4))
sns.barplot(data=df2, y='destinos', x='custo_medio', palette='magma')
plt.title('Top 3 Destinos com Maior Custo Médio')
plt.xlabel('Custo Médio (R$)')
plt.ylabel('Destinos')
plt.show()

# 3. A viagem de maior duração e seu custo total?
q3 = "SELECT id_viagem, duracao_dias, valor_total FROM silver_viagem ORDER BY duracao_dias DESC LIMIT 1"
df3 = pd.read_sql(q3, conexao)
display(df3)

### Parte 2: Construção da Camada Gold Agregada
Criaremos uma VIEW agregando informações através de JOIN e agrupamento (GROUP BY) das 3 tabelas principais.

In [ ]:
# Criando a camada Gold agregada (JOIN + GROUP BY) no banco de dados
sql_gold = """
CREATE OR REPLACE VIEW gold_resumo_viagens AS
SELECT 
    v.id_viagem, v.nome_orgao_superior, v.valor_total,
    p.tipo_pagamento, p.valor as valor_pagamento, p.nome_orgao_pagador,
    t.meio_transporte, t.destino_uf
FROM silver_viagem v
JOIN silver_pagamento p ON v.id_viagem = p.id_viagem
JOIN silver_trecho t ON v.id_viagem = t.id_viagem;
"""
banco.executar(conexao, sql_gold)
print('Camada Gold (VIEW gold_resumo_viagens) criada com sucesso!')

### Parte 3: Camada Gold - Analises Agregadas

In [ ]:
# 4. Qual o tipo de pagamento com maior valor médio?
q4 = "SELECT tipo_pagamento, AVG(valor_pagamento) as valor_medio FROM gold_resumo_viagens GROUP BY tipo_pagamento ORDER BY valor_medio DESC LIMIT 5"
df4 = pd.read_sql(q4, conexao)
display(df4)

# 5. Qual o meio de transporte mais usado nos trechos?
q5 = "SELECT meio_transporte, COUNT(*) as qtd FROM gold_resumo_viagens GROUP BY meio_transporte ORDER BY qtd DESC"
df5 = pd.read_sql(q5, conexao)
display(df5)
plt.figure(figsize=(10,4))
sns.barplot(data=df5, x='meio_transporte', y='qtd', palette='coolwarm')
plt.title('Meios de Transporte Mais Utilizados')
plt.xlabel('Meio de Transporte')
plt.ylabel('Quantidade de Trechos')
plt.show()

# 6. Qual UF de destino aparece em mais trechos?
q6 = "SELECT destino_uf, COUNT(*) as qtd FROM gold_resumo_viagens GROUP BY destino_uf ORDER BY qtd DESC LIMIT 5"
df6 = pd.read_sql(q6, conexao)
display(df6)
plt.figure(figsize=(10,4))
sns.barplot(data=df6, x='destino_uf', y='qtd', palette='crest')
plt.title('Top 5 UFs de Destino')
plt.xlabel('UF (Unidade Federativa)')
plt.ylabel('Quantidade de Trechos')
plt.show()

# 7. Qual órgão pagou mais no total?
q7 = "SELECT nome_orgao_pagador, SUM(valor_pagamento) as total_pago FROM gold_resumo_viagens GROUP BY nome_orgao_pagador ORDER BY total_pago DESC LIMIT 5"
df7 = pd.read_sql(q7, conexao)
display(df7)
plt.figure(figsize=(10,4))
sns.barplot(data=df7, y='nome_orgao_pagador', x='total_pago', palette='rocket')
plt.title('Top 5 Órgãos Pagadores pelo Total Pago')
plt.xlabel('Valor Total Pago (R$)')
plt.ylabel('Órgão Pagador')
plt.show()